# Underlink walkthrough

This notebook repeats the main Underlink steps from the files in `data/processed`. It runs top to bottom in under two minutes.

It prints counts only. It does not print relay keys, coordinates or community names, except Ampilatwatja and Galiwinku, whose outages were publicly reported.

Every number here should match `outputs/public/numbers.json`, which is what the report quotes. The last cell checks that.

In [1]:
# Make the project's src/ folder importable, wherever Jupyter was started.
import json, sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from underlink import hazards, network as nw, readouts as ro
from underlink.config import P, PROCESSED, PUBLIC

N = json.loads((PUBLIC / "numbers.json").read_text())   # published numbers, for the check at the end

## 1. Load the radio backhaul graph

Each node is a licensed point-to-point radio site from the ACMA register. Each edge is a licensed link. A radio site within 10 km of a fibre town is treated as connected to fibre (a "root").

**Cycle rank** (links minus sites plus separate pieces) counts independent loops. A low number means most sites have only one way back to fibre.

*Limit:* we see licensed radio links only. Fibre routes, satellite links and unlicensed links are not in the register.

In [2]:
net = nw.load()
integ = nw.integrity(net)
labels = {"V": "radio sites", "E": "radio links", "C": "separate pieces", "cycle_rank_radio": "loops, radio only",
          "cycle_rank_with_fibre": "loops, with fibre joined", "roots": "sites treated as on fibre",
          "anchors": "fibre towns", "components_with_root": "pieces that reach fibre"}
pd.Series(integ, name="count").rename(labels).to_frame()

,count
radio sites,358
radio links,313
separate pieces,51
"loops, radio only",6
"loops, with fibre joined",20
sites treated as on fibre,41
fibre towns,49
pieces that reach fibre,27


## 2. Give each place a serving site and a chain class

The nearest licensed radio site within 10 km stands in for the site that serves a place. Then each place gets one of four classes:

- **at-anchor**: its site is next to fibre
- **radio-chain**: it reaches fibre through one or more radio relays
- **radio-island**: it has licensed radio links but no radio path to fibre that we can see
- **no-radio-site**: no licensed radio site within 10 km

"Larger places" are the 116 towns, major and minor communities and villages in the NTG 2021 register.

*Limit:* the nearest site is a stand-in. It may not be the site that actually serves the place. The 10 km radii are assumptions, and the pipeline reruns everything at 5 and 15 km.

In [3]:
places = pd.read_csv(PROCESSED / "places.csv")
places = nw.assign_end_sites(places, net)
c = nw.classify(places, net)
L = c[c.larger]
L.chain_class.value_counts().rename("larger places").to_frame()

,larger places
chain_class,
no-radio-site,45
at-anchor,34
radio-chain,23
radio-island,14


## 3. Single points of failure

For a radio-chain place, we find the relays that lie on **every** path from its site to fibre. These are the site's dominators in graph terms. If one of them fails, the place loses its path to fibre. We call them single-point relays.

In [4]:
rc = L[L.chain_class == "radio-chain"]
summary = {
    "larger radio-chain places": len(rc),
    "depend on at least 1 single-point relay": int((rc.n_spof >= 1).sum()),
    "depend on 3 or more": int((rc.n_spof >= 3).sum()),
    "median hops to fibre": float(rc.hops.median()),
    "most hops to fibre": float(rc.hops.max()),
    "people in places with at least 1": int(rc[rc.n_spof >= 1].population_2020.sum()),
}
pd.Series(summary, name="value", dtype=object).to_frame()

,value
larger radio-chain places,23
depend on at least 1 single-point relay,18
depend on 3 or more,12
median hops to fibre,4.0
most hops to fibre,6.0
people in places with at least 1,7107


Two places whose outages were in the news. Both sit at the end of six-hop radio chains.

In [5]:
named = c[c.name.isin(["Ampilatwatja", "Galiwinku"])][["name", "chain_class", "hops", "n_spof"]]
named.rename(columns={"hops": "hops to fibre", "n_spof": "single-point relays"}).set_index("name")

,chain_class,hops to fibre,single-point relays
name,,,
Ampilatwatja,radio-chain,6.0,5
Galiwinku,radio-chain,6.0,5


In [6]:
# One row per relay that is a single point of failure for at least one place.
relays = nw.relay_table(c, net)
relays["power_class"] = relays.site_key.map(ro.power_classes(net))
print("single-point relays:", len(relays))
relays.power_class.value_counts().rename("relays").to_frame()

single-point relays: 68


,relays
power_class,
P0_unknown,49
P4_depot_within_150km,19


**Power classes.** P1 to P3 mean a published Mobile Network Hardening Program upgrade within 3 km. P4 means within 150 km of a Telstra portable-generator depot. P0 means unknown.

*Limit:* P0 means the battery hours are not published. It does not mean there is no battery. None of the single-point relays has published backup hours, which is our first data request.

## 4. Replay one past storm

We take the track of Tropical Cyclone Lam (2015), remove every relay within 100 km of it, and trace every place again. A place is **exposed** if it loses its path to fibre. It is **upstream only** if neither the place nor its own site was near the track, so it was cut off by a relay somewhere else.

*Limit:* this shows exposure. It does not say what failed in that storm.

In [7]:
tracks = hazards.load_tracks()
R = ro.replay(c, net, tracks, "AU201415_13U", 100, cyclone_only=False)
RL = R[R.larger]
pd.Series({
    "relays in the footprint": int(R.footprint_relays.iloc[0]),
    "larger places exposed": int(RL.exposed.sum()),
    "own site or place in the footprint": int(RL.direct.sum()),
    "cut off by an upstream relay only": int(RL.upstream_only.sum()),
}, name="Lam 2015, 100 km").to_frame()

,"Lam 2015, 100 km"
relays in the footprint,54
larger places exposed,6
own site or place in the footprint,3
cut off by an upstream relay only,3


## 5. Repair window: January against July

For each radio-chain place, we estimate days to restore the slowest element on its chain, split into three parts:

- **base**: half a day to diagnose and fix on site
- **travel**: from the nearest crew town, by road
- **access**: in the wet season, a relay more than 10 km from a sealed road waits 30 days for access

*Limit:* every input is an assumption in `config/params.yaml`. Read the result as where the time goes, not as a forecast.

In [8]:
jan = ro.place_repair(c, net, 1).merge(c[["place_key", "larger"]], on="place_key")
jul = ro.place_repair(c, net, 7).merge(c[["place_key", "larger"]], on="place_key")
jan, jul = jan[jan.larger], jul[jul.larger]
order = ["<1 d", "1-3 d", "3-14 d", ">14 d"]
bands = pd.DataFrame({"January": jan.band.value_counts(), "July": jul.band.value_counts()}).reindex(order).fillna(0).astype(int)
print(f"median days: January {jan.total.median():.1f}, July {jul.total.median():.1f}")
bands

median days: January 30.8, July 0.9


,January,July
band,,
<1 d,6,17
1-3 d,0,6
3-14 d,0,0
>14 d,17,0


## 6. What still works

For each larger place, we count fallbacks within 3 km that do not depend on the mobile tower: satellite Wi-Fi phones and STAND Sky Muster sites. We also count other carriers' mobile sites within 10 km, which could carry a 000 voice call.

*Limit:* payphones are not counted as independent, because some share the exchange. We have not checked any of this on the ground.

In [9]:
fb = ro.fallbacks(c).merge(c[["place_key", "larger", "chain_class"]], on="place_key")
fL = fb[fb.larger]
pd.Series({
    "larger places": len(fL),
    "with an independent fallback within 3 km": int(fL.independent_fallback.sum()),
    "with another carrier's site within 10 km": int((fL.other_carrier_site > 0).sum()),
    "radio-chain places with an independent fallback": int(((fL.chain_class == "radio-chain") & fL.independent_fallback).sum()),
}, name="count").to_frame()

,count
larger places,116
with an independent fallback within 3 km,48
with another carrier's site within 10 km,21
radio-chain places with an independent fallback,8


## 7. Check against the published numbers

The report quotes `outputs/public/numbers.json`. This cell fails if the notebook and that file disagree.

In [10]:
lam = next(e for e in N["replay"]["primary"] if e["event"] == "Lam 2015")
checks = {
    "sites": (integ["V"], N["network"]["V"]),
    "links": (integ["E"], N["network"]["E"]),
    "radio-chain places": (len(rc), N["chains"]["radio_chain_places"]),
    ">=1 single-point relay": (int((rc.n_spof >= 1).sum()), N["chains"]["with_ge1_spof"]),
    ">=3 single-point relays": (int((rc.n_spof >= 3).sum()), N["chains"]["with_ge3_spof"]),
    "single-point relays": (len(relays), N["relays"]["spof_relays"]),
    "Lam exposed": (int(RL.exposed.sum()), lam["exposed_larger"]),
    "January >14 d": (int((jan.band == ">14 d").sum()), N["repair"]["jan_bands"].get(">14 d", 0)),
    "independent fallback": (int(fL.independent_fallback.sum()), N["fallbacks"]["larger_with_independent_fallback"]),
}
for k, (mine, pub) in checks.items():
    assert mine == pub, f"{k}: notebook {mine}, numbers.json {pub}"
print("All", len(checks), "checks match numbers.json.")

All 9 checks match numbers.json.
